# 00 Environment and configuration

目标：建立并验证项目路径、五份配置、公共无状态模块和原子写入能力。

- 输入：仓库根目录、`config/*.yaml`
- 输出：`data/audit/environment_versions.json`、运行摘要
- 主键：不适用
- 时间范围：1980-01 至 2024-12
- 前置依赖：四份最终规范、最新重构 PDF、真实 CSV 目录
- 本 Notebook 不负责：任何业务数据变换

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
from src.path_utils import ensure_project_directories

assert ROOT.name != "Kaniel_Rebuild"
assert (ROOT / "Data_processing" / "manual_processing").is_dir()
required_directories = [
    "notebooks", "src", "config", "data/interim", "data/processed", "data/audit",
    "data/processed/macro", "outputs/panels", "outputs/npz", "outputs/reports",
    "outputs/staging", "outputs/archive", "logs", "docs",
]
created = ensure_project_directories(ROOT, required_directories)
print([str(path.relative_to(ROOT)) for path in created])


In [ ]:
from src.config_utils import config_hash
from src.schema_utils import require_paths

expected_configs = [
    ROOT / "config" / "paths.yaml",
    ROOT / "config" / "parameters.yaml",
    ROOT / "config" / "fund_filter.yaml",
    ROOT / "config" / "feature_directions.yaml",
    ROOT / "config" / "schemas.yaml",
]
require_paths(expected_configs)
assert len(CONFIG["feature_directions"]["fund_directions"]) == 13
stock_directions = CONFIG["feature_directions"]["stock_directions"]
assert stock_directions == {} or (
    len(stock_directions) == 46
    and set(int(value) for value in stock_directions.values()) <= {-1, 1}
    and str(CONFIG["feature_directions"].get("status", "")).startswith("frozen_1967_2016")
)
assert CONFIG["fund_filter"]["wficn_month"]["aggregation_statuses"] == ["pass", "unknown"]
assert PATHS["sentiment"].is_file() and PATHS["cfnai"].is_file()
assert CONFIG["path_options"]["sentiment"]["sheet_name"] == "DATA"
assert PARAMS["schema_version"] == "3.0.0"
assert PARAMS["pca_imputation"] == {
    "solver": "deterministic_direct_tsvd",
    "dtype": "float64",
    "n_components": 10,
    "minimum_observed_features": 10,
    "relative_singular_value_threshold": 0.05,
    "singular_value_rule": "retain s_i where s_i > rcond * s_max",
    "component_sign_rule": "largest_absolute_loading_positive",
    "observed_value_policy": "preserve_exactly",
    "missing_mask_policy": "preserve_exactly",
    "clipping": "none",
    "rounding": "none",
    "winsorization": "none",
    "linear_algebra_threads": 1,
}
assert PARAMS["label_name"] == "alpha"
assert PARAMS["label_semantic_name"] == "rolling_carhart_abnormal_return"
assert PARAMS["label_rename_only"] is True
assert PARAMS["label_values_changed"] is False
assert (
    PARAMS["core_feature_count"],
    PARAMS["macro_feature_count"],
    PARAMS["total_predictor_count"],
) == (59, 2, 61)
assert PARAMS["macro_feature_names"] == ["Sentiment", "CFNAI"]
assert PARAMS["macro_source_lag_months"] == {"Sentiment": 1, "CFNAI": 2}
for forbidden in {"alpha", "rret", "Sentiment", "CFNAI"}:
    assert forbidden not in CONFIG["feature_directions"]["feature_order"]
print({"config_hash": config_hash(ROOT / "config"), "config_files": [p.name for p in expected_configs]})


In [ ]:
import numpy as np
import pandas as pd
import openpyxl
from src.atomic_write import write_json_atomic
from src.audit_utils import environment_versions
from src.io_utils import clean_cusip8, normalize_management_name
from src.ranking import rank_to_half_interval, weighted_mean_same_valid_set
from src.runtime_paths import output_path
from src.schema_utils import safe_divide

assert clean_cusip8("123456") == "00123456"
assert normalize_management_name("Alpha Capital Management, LLC") == "alpha capital management"
assert np.isnan(safe_divide(1.0, 0.0))
assert weighted_mean_same_valid_set([1.0, np.nan, 3.0], [1.0, 100.0, 1.0]) == 2.0
ranked = rank_to_half_interval(pd.Series([1.0, 2.0, 2.0, 4.0]))
assert np.allclose(ranked.to_numpy(), [-0.5, 0.0, 0.0, 0.5])
assert tuple(int(part) for part in openpyxl.__version__.split(".")[:2]) >= (3, 0)

environment_path = output_path(
    ROOT, RUN_MODE, "00_environment_and_config", "data/audit/environment_versions.json"
)
write_json_atomic(environment_path, environment_versions())
print(json.loads(environment_path.read_text(encoding="utf-8")))


In [ ]:
from src.audit_utils import write_run_summary

summary_path = write_run_summary(
    ROOT,
    "00_environment_and_config",
    stage="00",
    started_at=STARTED_AT,
    input_files=[str(path.relative_to(ROOT)) for path in expected_configs],
    output_files=[str(environment_path.relative_to(ROOT))],
    input_row_counts={},
    output_row_counts={},
    primary_key=[],
    duplicate_key_count=0,
    excluded_row_count=0,
    exception_file=None,
    status="passed",
    notes=["Validated repository root; no nested project root was created."],
)
print(summary_path)
